# 🤖 ALOHA Bimanual Robot Test
구글 딥마인드의 `mujoco_menagerie`에서 공식 ALOHA 쌍팔 로봇 모델을 불러와 양팔을 움직여보는 테스트입니다.

In [1]:
# 필수 패키지 설치 및 딥마인드 공식 로봇 모델 저장소 다운로드
!pip install mujoco mediapy
!git clone https://github.com/google-deepmind/mujoco_menagerie.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 90.1 MB/s eta 0:00:00
Cloning into 'mujoco_menagerie'...
remote: Enumerating objects: 8018, done.
remote: Counting objects: 100% (1145/1145), done.
remote: Compressing objects: 100% (393/393), done.
remote: Total 8018 (delta 845), reused 820 (delta 751), pack-reused 6873 (from 2)
Receiving objects: 100% (8018/8018), 570.70 MiB | 20.84 MiB/s, done.
Resolving deltas: 100% (2693/2693), done.
Updating files: 100% (3040/3040), done.


In [2]:
import os
os.environ['MUJOCO_GL'] = 'egl'

import mujoco
import mediapy as media
import math

# Menagerie에서 ALOHA 씬(책상, 양팔, 카메라가 모두 세팅된 환경) 불러오기
xml_path = "mujoco_menagerie/aloha/scene.xml"
print("ALOHA 모델 로딩 중...")
model = mujoco.MjModel.from_xml_path(xml_path)
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, height=480, width=640)

# ALOHA 씬에 내장된 첫 번째 카메라 사용 (로봇 전체를 내려다보는 뷰)
cam_name = model.camera(0).name if model.ncam > 0 else None

frames = []
fps = 60
duration = 3.0  # 3초 분량

print("양팔 제어 및 렌더링 진행 중...")
for _ in range(int(fps * duration)):
    t = data.time

    # 로봇의 모든 모터(Actuator)에 제어 신호 보내기
    # 진폭을 0.2 라디안(약 11도)으로 작게 주어 책상에 강하게 부딪히지 않고 부드럽게 흔들리도록 설정합니다.
    for i in range(model.nu):
        data.ctrl[i] = math.sin(t * (2.0 + i * 0.1)) * 0.2

    mujoco.mj_step(model, data)
    renderer.update_scene(data, camera=cam_name)
    frames.append(renderer.render())

print("비디오 생성 완료!")
media.show_video(frames, fps=fps)


ALOHA 모델 로딩 중...
양팔 제어 및 렌더링 진행 중...
비디오 생성 완료!


This browser does not support the video tag.
